In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.linear_model    import LogisticRegression, LinearRegression
from sklearn.tree            import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report,
    ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error, r2_score
)
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

# Auto-detect dataset (works from any working directory)
POSSIBLE = [
    os.path.join(os.getcwd(), 'data', 'raw', 'diabetes_dataset.csv'),
    os.path.join(os.getcwd(), '..', 'data', 'raw', 'diabetes_dataset.csv'),
    '/mnt/user-data/uploads/diabetes_dataset.csv',
]
DATA_PATH = next((p for p in POSSIBLE if os.path.exists(p)), None)
assert DATA_PATH, "Dataset not found — place diabetes_dataset.csv in data/raw/"
print(f"Dataset: {DATA_PATH}")

BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(DATA_PATH), '..', '..'))
PROC_DIR   = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("Setup complete ✓")

Dataset: d:\Diabetes_Health_Indicator\diabetes-health-indicators-ml\notebooks\..\data\raw\diabetes_dataset.csv
Setup complete ✓


In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

Shape: (100000, 31)


,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,48,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 31 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   age                                 100000 non-null  int64  
 1   gender                              100000 non-null  object 
 2   ethnicity                           100000 non-null  object 
 3   education_level                     100000 non-null  object 
 4   income_level                        100000 non-null  object 
 5   employment_status                   100000 non-null  object 
 6   smoking_status                      100000 non-null  object 
 7   alcohol_consumption_per_week        100000 non-null  int64  
 8   physical_activity_minutes_per_week  100000 non-null  int64  
 9   diet_score                          100000 non-null  float64
 10  sleep_hours_per_day                 100000 non-null  float64
 11  screen_time_hours_per_day  

In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,100000.0,50.12,15.60,18.00,39.00,50.00,61.00,90.00
alcohol_consumption_per_week,100000.0,2.00,1.42,0.00,1.00,2.00,3.00,10.00
physical_activity_minutes_per_week,100000.0,118.91,84.41,0.00,57.00,100.00,160.00,833.00
diet_score,100000.0,5.99,1.78,0.00,4.80,6.00,7.20,10.00
sleep_hours_per_day,100000.0,7.00,1.09,3.00,6.30,7.00,7.70,10.00
screen_time_hours_per_day,100000.0,6.00,2.47,0.50,4.30,6.00,7.70,16.80
family_history_diabetes,100000.0,0.22,0.41,0.00,0.00,0.00,0.00,1.00
hypertension_history,100000.0,0.25,0.43,0.00,0.00,0.00,1.00,1.00
cardiovascular_history,100000.0,0.08,0.27,0.00,0.00,0.00,0.00,1.00
bmi,100000.0,25.61,3.59,15.00,23.20,25.60,28.00,39.20


In [5]:
print("Missing values:", df.isnull().sum().sum(), "(none expected)")
print()
print("=== diagnosed_diabetes ===")
print(df['diagnosed_diabetes'].value_counts())
print()
print("=== diabetes_stage ===")
print(df['diabetes_stage'].value_counts())
print()
print("=== diabetes_risk_score ===")
print(df['diabetes_risk_score'].describe())

Missing values: 0 (none expected)

=== diagnosed_diabetes ===
diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64

=== diabetes_stage ===
diabetes_stage
Type 2          59774
Pre-Diabetes    31845
No Diabetes      7981
Gestational       278
Type 1            122
Name: count, dtype: int64

=== diabetes_risk_score ===
count    100000.000000
mean         30.222362
std           9.061505
min           2.700000
25%          23.800000
50%          29.000000
75%          35.600000
max          67.200000
Name: diabetes_risk_score, dtype: float64


In [6]:
CATEGORICAL_COLS = ['gender','ethnicity','education_level',
                    'income_level','employment_status','smoking_status']

NUMERICAL_COLS = [
    'age','alcohol_consumption_per_week','physical_activity_minutes_per_week',
    'diet_score','sleep_hours_per_day','screen_time_hours_per_day',
    'bmi','waist_to_hip_ratio','systolic_bp','diastolic_bp','heart_rate',
    'cholesterol_total','hdl_cholesterol','ldl_cholesterol','triglycerides',
    'glucose_fasting','glucose_postprandial','insulin_level','hba1c'
]

# Impute missing values
for col in NUMERICAL_COLS:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
for col in CATEGORICAL_COLS:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
print("Missing after imputation:", df.isnull().sum().sum())

# Label encode categoricals
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# Encode multiclass target
le_stage = LabelEncoder()
df['diabetes_stage'] = le_stage.fit_transform(df['diabetes_stage'].astype(str))
encoders['diabetes_stage'] = le_stage

print("Encoded diabetes_stage classes:", list(le_stage.classes_))
print("Preprocessing complete ✓")
df.head()

Missing after imputation: 0
Encoded diabetes_stage classes: ['Gestational', 'No Diabetes', 'Pre-Diabetes', 'Type 1', 'Type 2']
Preprocessing complete ✓


,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,1,0,1,2,0,2,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,4,1
1,48,0,4,1,3,0,1,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,1,0
2,60,1,2,1,3,3,2,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,4,1
3,74,0,1,1,1,1,2,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,4,1
4,46,1,4,0,3,1,2,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,4,1


In [7]:
# Save processed data
out_csv = os.path.join(PROC_DIR, 'diabetes_processed.csv')
df.to_csv(out_csv, index=False)
joblib.dump(encoders, os.path.join(PROC_DIR, 'label_encoders.joblib'))
print(f"Saved: {out_csv} ✓")

Saved: d:\Diabetes_Health_Indicator\diabetes-health-indicators-ml\data\processed\diabetes_processed.csv ✓


In [8]:
# Class balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['diagnosed_diabetes'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#2196F3','#F44336'], rot=0)
axes[0].set_title('Class Balance — diagnosed_diabetes')
axes[0].set_xticklabels(['No Diabetes','Diabetes'])
axes[0].set_ylabel('Count')

stage_idx    = df['diabetes_stage'].value_counts()
stage_labels = [le_stage.classes_[i] for i in stage_idx.index]
axes[1].bar(range(len(stage_labels)), stage_idx.values,
            color=['#4CAF50','#FF9800','#F44336','#9C27B0','#2196F3'][:len(stage_labels)])
axes[1].set_xticks(range(len(stage_labels)))
axes[1].set_xticklabels(stage_labels, rotation=20)
axes[1].set_title('Class Balance — diabetes_stage')
plt.tight_layout()
plt.show()

In [9]:
# Feature distributions
key_features = ['age','bmi','hba1c','glucose_fasting',
                'glucose_postprandial','insulin_level','cholesterol_total','triglycerides']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), key_features):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=10)
plt.suptitle('Key Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [10]:
# Boxplots vs binary target
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), key_features):
    df.boxplot(column=col, by='diagnosed_diabetes', ax=ax)
    ax.set_title(col); ax.set_xlabel('0=No Diabetes  1=Diabetes')
plt.suptitle('Features by Diabetes Diagnosis')
plt.tight_layout()
plt.show()

In [11]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr_cols = NUMERICAL_COLS + ['diagnosed_diabetes','diabetes_risk_score']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            linewidths=0.3, cbar_kws={'shrink':0.8}, annot=False)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [12]:
# Top correlations with binary target
top_corr = (corr['diagnosed_diabetes'].drop('diagnosed_diabetes')
            .abs().sort_values(ascending=True).tail(12))
plt.figure(figsize=(8, 5))
top_corr.plot(kind='barh', color='steelblue')
plt.title('Top Features Correlated with diagnosed_diabetes')
plt.xlabel('|Correlation|')
plt.tight_layout()
plt.show()
print(top_corr.sort_values(ascending=False).round(3).to_string())

hba1c                                 0.679
glucose_postprandial                  0.630
glucose_fasting                       0.511
diabetes_risk_score                   0.277
age                                   0.138
physical_activity_minutes_per_week    0.101
bmi                                   0.097
systolic_bp                           0.095
waist_to_hip_ratio                    0.079
ldl_cholesterol                       0.067
cholesterol_total                     0.058
insulin_level                         0.058


In [13]:
# Risk score distribution
plt.figure(figsize=(10, 4))
plt.hist(df['diabetes_risk_score'], bins=60, color='darkorange', edgecolor='white', alpha=0.85)
mu = df['diabetes_risk_score'].mean()
plt.axvline(mu, color='red', ls='--', lw=2, label=f'Mean={mu:.1f}')
plt.title('Distribution — diabetes_risk_score')
plt.xlabel('Risk Score'); plt.ylabel('Frequency'); plt.legend()
plt.tight_layout()
plt.show()

In [14]:
T_BIN   = 'diagnosed_diabetes'
T_MULTI = 'diabetes_stage'
T_REG   = 'diabetes_risk_score'

X      = df.drop(columns=[T_BIN, T_MULTI, T_REG])
y_bin  = df[T_BIN]
y_multi= df[T_MULTI]
y_reg  = df[T_REG]

Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X, y_bin,   test_size=0.2, stratify=y_bin,   random_state=RANDOM_STATE)
Xtr_m, Xte_m, ytr_m, yte_m = train_test_split(X, y_multi, test_size=0.2, stratify=y_multi, random_state=RANDOM_STATE)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y_reg,   test_size=0.2,                   random_state=RANDOM_STATE)

def scale(Xtr, Xte):
    sc = StandardScaler()
    return (pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns),
            pd.DataFrame(sc.transform(Xte),     columns=Xte.columns), sc)

Xtr_bs, Xte_bs, sc_b = scale(Xtr_b, Xte_b)
Xtr_ms, Xte_ms, sc_m = scale(Xtr_m, Xte_m)
Xtr_rs, Xte_rs, sc_r = scale(Xtr_r, Xte_r)

print(f"Binary     train/test: {Xtr_b.shape} / {Xte_b.shape}")
print(f"Multiclass train/test: {Xtr_m.shape} / {Xte_m.shape}")
print(f"Regression train/test: {Xtr_r.shape} / {Xte_r.shape}")
print(f"Class balance (binary train): {dict(ytr_b.value_counts())}")

Binary     train/test: (80000, 28) / (20000, 28)
Multiclass train/test: (80000, 28) / (20000, 28)
Regression train/test: (80000, 28) / (20000, 28)
Class balance (binary train): {1: np.int64(47998), 0: np.int64(32002)}


In [15]:
bin_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}
bin_results, bin_probas, bin_trained = {}, {}, {}

for name, model in bin_models.items():
    model.fit(Xtr_bs, ytr_b)
    yp  = model.predict(Xte_bs)
    ypr = model.predict_proba(Xte_bs)[:,1] if hasattr(model,'predict_proba') else None
    bin_probas[name]  = ypr
    bin_trained[name] = model
    bin_results[name] = {
        'Accuracy':  accuracy_score(yte_b, yp),
        'Precision': precision_score(yte_b, yp, zero_division=0),
        'Recall':    recall_score(yte_b, yp, zero_division=0),
        'F1':        f1_score(yte_b, yp, zero_division=0),
        'ROC_AUC':   roc_auc_score(yte_b, ypr) if ypr is not None else None,
    }
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    print(classification_report(yte_b, yp, target_names=['No Diabetes','Diabetes']))

pd.DataFrame(bin_results).T.round(4)


Logistic Regression
              precision    recall  f1-score   support

 No Diabetes       0.84      0.81      0.82      8000
    Diabetes       0.87      0.90      0.88     12000

    accuracy                           0.86     20000
   macro avg       0.86      0.85      0.85     20000
weighted avg       0.86      0.86      0.86     20000


Decision Tree
              precision    recall  f1-score   support

 No Diabetes       0.84      0.81      0.82      8000
    Diabetes       0.88      0.89      0.89     12000

    accuracy                           0.86     20000
   macro avg       0.86      0.85      0.85     20000
weighted avg       0.86      0.86      0.86     20000


KNN
              precision    recall  f1-score   support

 No Diabetes       0.75      0.77      0.76      8000
    Diabetes       0.84      0.83      0.84     12000

    accuracy                           0.81     20000
   macro avg       0.80      0.80      0.80     20000
weighted avg       0.81      0.81

,Accuracy,Precision,Recall,F1,ROC_AUC
Logistic Regression,0.8604,0.8750,0.8952,0.8850,0.9338
Decision Tree,0.8613,0.8768,0.8946,0.8856,0.8530
KNN,0.8057,0.8434,0.8303,0.8368,0.8726


In [16]:
# ROC Curves
plt.figure(figsize=(8, 6))
for name, proba in bin_probas.items():
    if proba is None: continue
    fpr, tpr, _ = roc_curve(yte_b, proba)
    auc = roc_auc_score(yte_b, proba)
    plt.plot(fpr, tpr, lw=2, label=f'{name}  (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Binary Classification')
plt.legend(loc='lower right'); plt.tight_layout()
plt.show()

In [17]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, bin_trained.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, Xte_bs, yte_b, display_labels=['No Diabetes','Diabetes'],
        cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(name)
plt.suptitle('Confusion Matrices — Binary', fontsize=13)
plt.tight_layout()
plt.show()

In [18]:
class_names = list(le_stage.classes_)
print("Classes:", class_names)

multi_models = {
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}
multi_results, multi_trained = {}, {}

for name, model in multi_models.items():
    model.fit(Xtr_ms, ytr_m)
    yp  = model.predict(Xte_ms)
    acc = accuracy_score(yte_m, yp)
    mf1 = f1_score(yte_m, yp, average='macro', zero_division=0)
    multi_results[name] = {'Accuracy': acc, 'Macro_F1': mf1}
    multi_trained[name] = model
    print(f"{name}: Accuracy={acc:.4f}  Macro-F1={mf1:.4f}")

pd.DataFrame(multi_results).T.round(4)

Classes: ['Gestational', 'No Diabetes', 'Pre-Diabetes', 'Type 1', 'Type 2']
Decision Tree: Accuracy=0.8552  Macro-F1=0.5141
Logistic Regression: Accuracy=0.8192  Macro-F1=0.4658
KNN: Accuracy=0.7557  Macro-F1=0.4141


,Accuracy,Macro_F1
Decision Tree,0.8552,0.5141
Logistic Regression,0.8192,0.4658
KNN,0.7556,0.4141


In [19]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, multi_trained.items()):
    r = multi_results[name]
    ConfusionMatrixDisplay.from_estimator(
        model, Xte_ms, yte_m, display_labels=class_names,
        cmap='Blues', ax=ax, colorbar=False, xticks_rotation=25)
    ax.set_title(f"{name}\nAcc={r['Accuracy']:.3f}  F1={r['Macro_F1']:.3f}")
plt.suptitle('Multiclass Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig('multiclass_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [20]:
reg_models = {
    'Linear Regression':       LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=RANDOM_STATE),
}
reg_results, reg_preds, reg_trained = {}, {}, {}

for name, model in reg_models.items():
    model.fit(Xtr_rs, ytr_r)
    yp   = model.predict(Xte_rs)
    mae  = mean_absolute_error(yte_r, yp)
    rmse = np.sqrt(mean_squared_error(yte_r, yp))
    r2   = r2_score(yte_r, yp)
    reg_results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    reg_preds[name]   = yp
    reg_trained[name] = model
    print(f"{name}: MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}")

pd.DataFrame(reg_results).T.round(4)

Linear Regression: MAE=0.439  RMSE=0.745  R²=0.9933
Decision Tree Regressor: MAE=1.247  RMSE=1.595  R²=0.9692


,MAE,RMSE,R2
Linear Regression,0.4390,0.7446,0.9933
Decision Tree Regressor,1.2474,1.5949,0.9692


In [21]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, yp) in zip(axes, reg_preds.items()):
    ax.scatter(yte_r, yp, alpha=0.15, s=5, color='steelblue')
    lo = min(float(yte_r.min()), float(yp.min()))
    hi = max(float(yte_r.max()), float(yp.max()))
    ax.plot([lo,hi],[lo,hi],'r--',lw=1.5,label='Perfect fit')
    ax.set_title(f"{name}  (R²={reg_results[name]['R2']:.4f})")
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted'); ax.legend()
plt.suptitle('Actual vs Predicted — Risk Score')
plt.tight_layout()
plt.savefig('regression_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

In [22]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("[Tuning] Logistic Regression (Binary) ...")
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    {'C': [0.01, 0.1, 1, 10]}, cv=cv5, scoring='roc_auc', n_jobs=-1)
lr_gs.fit(Xtr_bs, ytr_b)
print(f"  Best params: {lr_gs.best_params_}  CV AUC={lr_gs.best_score_:.4f}")

print("\n[Tuning] Decision Tree (Binary) ...")
dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    {'max_depth':[5,10,20,None], 'min_samples_leaf':[1,5,10]},
    cv=cv5, scoring='f1', n_jobs=-1)
dt_gs.fit(Xtr_bs, ytr_b)
print(f"  Best params: {dt_gs.best_params_}  CV F1={dt_gs.best_score_:.4f}")

[Tuning] Logistic Regression (Binary) ...
  Best params: {'C': 10}  CV AUC=0.9340

[Tuning] Decision Tree (Binary) ...
  Best params: {'max_depth': 5, 'min_samples_leaf': 10}  CV F1=0.9299


In [23]:
# Compare baseline vs tuned
rows = []
for label, gs in [('LR (tuned)', lr_gs), ('DT (tuned)', dt_gs)]:
    yp   = gs.best_estimator_.predict(Xte_bs)
    yprb = gs.best_estimator_.predict_proba(Xte_bs)[:,1]
    rows.append({'Model': label,
                 'Accuracy': accuracy_score(yte_b, yp),
                 'F1':       f1_score(yte_b, yp, zero_division=0),
                 'ROC_AUC':  roc_auc_score(yte_b, yprb)})

tuned_df = pd.DataFrame(rows).set_index('Model').round(4)
print("Tuned Results:")
print(tuned_df.to_string())
tuned_df

Tuned Results:
            Accuracy      F1  ROC_AUC
Model                                
LR (tuned)    0.8604  0.8850   0.9338
DT (tuned)    0.9199  0.9285   0.9431


,Accuracy,F1,ROC_AUC
Model,,,
LR (tuned),0.8604,0.8850,0.9338
DT (tuned),0.9199,0.9285,0.9431


In [24]:
best_bin   = lr_gs.best_estimator_
best_multi_name = max(multi_results, key=lambda k: multi_results[k]['Macro_F1'])
best_reg_name   = max(reg_results,   key=lambda k: reg_results[k]['R2'])
best_multi = multi_trained[best_multi_name]
best_reg   = reg_trained[best_reg_name]

joblib.dump({'model': best_bin,   'scaler': sc_b, 'features': list(X.columns)},
            os.path.join(MODELS_DIR, 'binary_model.joblib'))
joblib.dump({'model': best_multi, 'scaler': sc_m, 'features': list(X.columns),
             'classes': class_names},
            os.path.join(MODELS_DIR, 'multiclass_model.joblib'))
joblib.dump({'model': best_reg,   'scaler': sc_r, 'features': list(X.columns)},
            os.path.join(MODELS_DIR, 'regression_model.joblib'))

print("✅ Models saved to:", MODELS_DIR)
print(f"  Binary     → Logistic Regression (tuned)")
print(f"  Multiclass → {best_multi_name}")
print(f"  Regression → {best_reg_name}")

✅ Models saved to: d:\Diabetes_Health_Indicator\diabetes-health-indicators-ml\models
  Binary     → Logistic Regression (tuned)
  Multiclass → Decision Tree
  Regression → Linear Regression
